In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/ricedisease-traindataset/Rice Disease Dataset/Leaf Scald/LS_277.jpg
/kaggle/input/ricedisease-traindataset/Rice Disease Dataset/Leaf Scald/Augmented_0_8640.jpeg
/kaggle/input/ricedisease-traindataset/Rice Disease Dataset/Leaf Scald/leaf_scald1648.jpg
/kaggle/input/ricedisease-traindataset/Rice Disease Dataset/Leaf Scald/leaf_scald597.jpg
/kaggle/input/ricedisease-traindataset/Rice Disease Dataset/Leaf Scald/leaf_scald1387.jpg
/kaggle/input/ricedisease-traindataset/Rice Disease Dataset/Leaf Scald/Augmented_0_4348.jpeg
/kaggle/input/ricedisease-traindataset/Rice Disease Dataset/Leaf Scald/leaf_scald1253.jpg
/kaggle/input/ricedisease-traindataset/Rice Disease Dataset/Leaf Scald/leaf_scald1499.jpg
/kaggle/input/ricedisease-traindataset/Rice Disease Dataset/Leaf Scald/LS_298.jpg
/kaggle/input/ricedisease-traindataset/Rice Disease Dataset/Leaf Scald/Augmented_0_8126.jpeg
/kaggle/input/ricedisease-traindataset/Rice Disease Dataset/Leaf Scald/leaf_scald873.jpg
/kaggle/input/riced

In [2]:
!pip install timm

In [1]:
import torch
import torch.nn as nn
import timm
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, Subset, ConcatDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import os
from PIL import Image
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [2]:
class_mapping = {
    "Bacterial Leaf Blight": 0,
    "Bacterial Streak": 1,
    "Bakanae": 2,
    "Brown Spot": 3,
    "False Smut": 4,
    "Grassy Stunt Virus": 5,
    "Healthy Leaf": 6,
    "Hispa": 7,
    "Leaf Blast": 8,
    "Leaf Scald": 9,
    "Leaf Smut": 10,
    "Narrow Brown Spot": 11,
    "Neck Blast": 12,
    "Ragged Stunt Virus": 13,
    "Sheath Blight": 14,
    "Sheath Rot": 15,
    "Stem Rot": 16,
    "Tungro": 17,
    "Insect Affected": 18,
}

In [3]:
train_transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

In [4]:
class PlantDocDataset(Dataset):
    def __init__(self, root, transform=None):
        self.samples = []
        self.transform = transform

        for class_name in os.listdir(root):
            if class_name in class_mapping:
                class_path = os.path.join(root, class_name)
                for img in os.listdir(class_path):
                    self.samples.append(
                        (os.path.join(class_path, img),
                         class_mapping[class_name])
                    )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [5]:
plantdoc_root = "/kaggle/input/ricedisease-traindataset/Rice Disease Dataset"

full_pd_dataset = PlantDocDataset(
    plantdoc_root,
    transform=train_transform
)

indices = list(range(len(full_pd_dataset)))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)

pd_train_subset = Subset(full_pd_dataset, train_idx)

pd_val_dataset = PlantDocDataset(
    plantdoc_root,
    transform=val_transform
)

pd_val_subset = Subset(pd_val_dataset, val_idx)

print("Rice dataset train size:", len(pd_train_subset))
print("Rice dataset val size:", len(pd_val_subset))

Rice dataset train size: 36343
Rice dataset val size: 9086


In [6]:
# joint_train_dataset = ConcatDataset([pv_dataset, pd_train_subset])
joint_train_dataset = pd_train_subset


train_loader = DataLoader(
    joint_train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    pd_val_subset,
    batch_size=16,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

num_classes = 19

In [7]:
model = timm.create_model(
    "vit_base_patch14_dinov2.lvd142m",
    pretrained=True,
    img_size=224
)

for param in model.parameters():
    param.requires_grad = False

for param in model.blocks[-4:].parameters():
    param.requires_grad = True

in_features = model.num_features

model.head = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_features, num_classes)
)

model = model.to(device)

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

In [8]:
from collections import Counter

train_labels = [label for _, label in pd_train_subset]
class_counts = Counter(train_labels)

weights = [1.0 / class_counts[i] for i in range(num_classes)]
weights = torch.tensor(weights).float().to(device)

criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.02)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=30
)

In [9]:
from torch.amp import GradScaler, autocast

def train_model(model, train_loader, val_loader, epochs=30, patience=10):

    scaler = GradScaler("cuda")
    best_acc = 0
    early_stop = 0

    for epoch in range(epochs):

        model.train()
        running_loss = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            with autocast("cuda"):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()

        model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in val_loader:

                images = images.to(device)
                labels = labels.to(device)

                with autocast("cuda"):
                    outputs = model(images)

                _, preds = torch.max(outputs,1)

                total += labels.size(0)
                correct += (preds==labels).sum().item()

        val_acc = correct / total

        print(f"Epoch {epoch+1}: Loss={running_loss/len(train_loader):.4f} | Val Acc={val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "best_dino_finetuned.pth")
            early_stop = 0
        else:
            early_stop += 1

        if early_stop >= patience:
            print("Early stopping triggered.")
            break

        scheduler.step()

    print("Best Validation Accuracy:", best_acc)

In [10]:
train_model(model, train_loader, val_loader, epochs=30)

Epoch 1: Loss=1.9761 | Val Acc=0.8196
Epoch 2: Loss=1.5991 | Val Acc=0.8731
Epoch 3: Loss=1.5254 | Val Acc=0.8921
Epoch 4: Loss=1.4904 | Val Acc=0.9076
Epoch 5: Loss=1.4626 | Val Acc=0.9072
Epoch 6: Loss=1.4219 | Val Acc=0.9073
Epoch 7: Loss=1.4182 | Val Acc=0.9158
Epoch 8: Loss=1.3999 | Val Acc=0.9176
Epoch 9: Loss=1.3951 | Val Acc=0.9202
Epoch 10: Loss=1.3889 | Val Acc=0.9166
Epoch 11: Loss=1.3817 | Val Acc=0.9275
Epoch 12: Loss=1.3714 | Val Acc=0.9241
Epoch 13: Loss=1.3648 | Val Acc=0.9318
Epoch 14: Loss=1.3589 | Val Acc=0.9332
Epoch 15: Loss=1.3599 | Val Acc=0.9302
Epoch 16: Loss=1.3549 | Val Acc=0.9285
Epoch 17: Loss=1.3541 | Val Acc=0.9341
Epoch 18: Loss=1.3474 | Val Acc=0.9335
Epoch 19: Loss=1.3483 | Val Acc=0.9325
Epoch 20: Loss=1.3480 | Val Acc=0.9342
Epoch 21: Loss=1.3494 | Val Acc=0.9367
Epoch 22: Loss=1.3415 | Val Acc=0.9378
Epoch 23: Loss=1.3403 | Val Acc=0.9375
Epoch 24: Loss=1.3378 | Val Acc=0.9365
Epoch 25: Loss=1.3403 | Val Acc=0.9372
Epoch 26: Loss=1.3419 | Val Acc=0.

In [11]:
test_transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

plantdoc_test_root = "/kaggle/input/datasets/vishnuawasthi/rice-test-data/Rice Disease Dataset Test"

pd_test_dataset = PlantDocDataset(
    plantdoc_test_root,
    transform=test_transform
)

test_loader = DataLoader(
    pd_test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

print("Rice dataset Test size:", len(pd_test_dataset))

Rice dataset Test size: 465


In [12]:
import timm
import torch.nn as nn

num_classes = 19

model = timm.create_model(
    "vit_base_patch14_dinov2.lvd142m",
    pretrained=False,
    img_size=224
)

in_features = model.num_features
model.head = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_features, num_classes)
)
model.load_state_dict(
    torch.load("best_dino_finetuned.pth", map_location=device)
)

model = model.to(device)
model.eval()

print("Best DINO model loaded successfully.")

Best DINO model loaded successfully.


In [13]:
from sklearn.metrics import confusion_matrix, classification_report

correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = correct / total

print("\n==============================")
print("DINOv2 Test Accuracy:", test_acc)
print("==============================")

cm = confusion_matrix(all_labels, all_preds)

print("\nConfusion Matrix:")
print(cm)

class_names = [
    "Bacterial Leaf Blight",
    "Bacterial Streak",
    "Bakanae",
    "Brown Spot",
    "False Smut",
    "Grassy Stunt Virus",
    "Healthy Leaf",
    "Hispa",
    "Leaf Blast",
    "Leaf Scald",
    "Leaf Smut",
    "Narrow Brown Spot",
    "Neck Blast",
    "Ragged Stunt Virus",
    "Sheath Blight",
    "Sheath Rot",
    "Stem Rot",
    "Tungro",
    "Insect Affected",
]

print("\nPer Class Accuracy:")

for i, class_name in enumerate(class_names):
    class_total = cm[i].sum()
    class_correct = cm[i][i]

    acc = class_correct / class_total if class_total > 0 else 0
    print(f"{class_name}: {acc:.4f}")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))


DINOv2 Test Accuracy: 0.946236559139785

Confusion Matrix:
[[29  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0]
 [ 0 15  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0 15  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0 23  0  0  0  0  0  0  0  7  0  0  0  0  0  0  0]
 [ 0  0  0  0 15  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  2  0  0 12  0  0  0  0  0  0  0  1  0  0  0  0  0]
 [ 0  0  0  0  0  0 30  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0 30  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0 29  0  0  0  0  0  0  0  0  1  0]
 [ 0  0  0  1  0  0  0  0  0 29  0  0  0  0  0  0  0  0  0]
 [ 1  0  0  3  0  0  0  0  0  0 26  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0 30  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0 30  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  1  0  0  0  0 14  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0 30  0  0  0  0]
 [ 5  0  0  0  0  0  0  0  0  0  0  0  0